In [1]:
import pandas as pd
import os
import glob
import numpy as np


In [2]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [3]:
folder_path = os.getcwd()
dataset_folder_path = os.path.join(folder_path, "..\\data_raw\\")
# Files inside dataset_raw
print(os.listdir(dataset_folder_path))


['CPI.xlsx', 'events.xlsx', 'interest_rates.xlsx', 'market_index', 'news.xlsx', 'sector_indices', 'WPI.xlsx']


### First we will clean the items which have multiple files such as market index and sector indexes

In [4]:
def files_merger(input_folder, output_file):
    # Get all CSV files
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

    dataframes = []

    for file in csv_files:
        print(f"Processing: {os.path.basename(file)}")

        df = pd.read_csv(file)

        # Standardize column names
        df.columns = (
            df.columns
            .str.strip()               # remove leading/trailing spaces
            .str.lower()               # lowercase
            .str.replace(r"\s+", "", regex=True)  # remove all spaces
        )

        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df["close"] = pd.to_numeric(df["close"].astype(str).str.replace(",", ""), errors="coerce")

        # Sort by date in ascending order
        dataframes.append(df)

    # Merge all files
    merged_df = pd.concat(dataframes, ignore_index=True, sort=False)

    # # Optional: remove duplicate rows
    merged_df = merged_df.drop_duplicates()
    final_df = merged_df[["date", "close"]]
    final_df = final_df.sort_values(by='date', ascending=True)

    # Optional: reset index
    final_df = final_df.reset_index(drop=True)

    # Save merged file
    output_file_path = f"{cwd}\\..\\data_clean\\{output_file}.csv"
    final_df.to_csv(output_file_path, index=False)

    print(f"Merged file saved as: {output_file}.csv")
    print(f"Total rows: {final_df.shape[0]}")
    print(f"Total columns: {len(final_df.columns)}")


In [5]:
# Here we are going to clean the datasets and merge them into single
# Current working directory
cwd = os.getcwd()

# Folder containing CSV files
input_folder = os.path.join(cwd, "..\\data_raw")
input_sector_folder = os.path.join(input_folder, "sector_indices")
input_folders_list = [
    "market_index",
    "nifty_auto",
    "nifty_bank",
    "nifty_financial_services",
    "nifty_fmcg",
    "nifty_it",
    "nifty_metal",
    "nifty_pharma",
    "nifty_realty",
    "nifty_energy"]

for folder in input_folders_list:
    if folder.startswith("nifty"):
        files_path = os.path.join(input_sector_folder, folder)
    else:
        files_path = os.path.join(input_folder, folder)

    files_merger(files_path, folder)

    print("-----------------------------------------")


Processing: NIFTY MIDCAP 150-01-01-2008-to-31-12-2008.csv
Processing: NIFTY MIDCAP 150-01-01-2009-to-31-12-2009.csv
Processing: NIFTY MIDCAP 150-01-01-2010-to-31-12-2010.csv
Processing: NIFTY MIDCAP 150-01-01-2011-to-31-12-2011.csv
Processing: NIFTY MIDCAP 150-01-01-2012-to-31-12-2012.csv
Processing: NIFTY MIDCAP 150-01-01-2013-to-31-12-2013.csv
Processing: NIFTY MIDCAP 150-01-01-2014-to-31-12-2014.csv
Processing: NIFTY MIDCAP 150-01-01-2015-to-31-12-2015.csv
Processing: NIFTY MIDCAP 150-01-01-2016-to-31-12-2016.csv
Processing: NIFTY MIDCAP 150-01-01-2017-to-31-12-2017.csv
Processing: NIFTY MIDCAP 150-01-01-2018-to-31-12-2018.csv
Processing: NIFTY MIDCAP 150-01-01-2019-to-31-12-2019.csv
Processing: NIFTY MIDCAP 150-01-01-2020-to-31-12-2020.csv
Processing: NIFTY MIDCAP 150-01-01-2021-to-31-12-2021.csv
Processing: NIFTY MIDCAP 150-01-01-2022-to-31-12-2022.csv
Processing: NIFTY MIDCAP 150-01-01-2023-to-31-12-2023.csv
Processing: NIFTY MIDCAP 150-01-01-2024-to-31-12-2024.csv
Processing: NI

Note that in some cases if date can't be parsed then it will left null which needs to be handled using excel

### Now to clean individual files which includes:
* Interest Rates
* CPI
* WPI
* Events
* News

### Cleaning CPI file

In [6]:
import re

# Read file (works for CSV or TSV)
cpi_file = os.path.join(input_folder, "CPI.xlsx")
df = pd.read_excel(cpi_file)

# Clean column names (just in case)
df.columns = [col.strip() for col in df.columns]
df.columns


Index(['Release date', 'Actual', 'Forecast', 'Previous'], dtype='object')

In [7]:
# Function to clean percentage values
def clean_percent(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    x = re.sub(r"%", "", x)
    try:
        return float(x)
    except BaseException:
        return None


# Clean numeric columns
for col in ["Actual", "Forecast", "Previous"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_percent)

df.head()


,Release date,Actual,Forecast,Previous
0,"Jun 12, 2026 (May)",0.0393,0.0400,0.0348
1,"May 12, 2026 (Apr)",0.0348,0.0380,0.0340
2,"Apr 13, 2026 (Mar)",0.0340,0.0348,0.0321
3,"Mar 12, 2026 (Feb)",0.0321,0.0310,0.0273
4,"Feb 12, 2026 (Jan)",0.0275,0.0240,0.0133


In [8]:
# Clean Release date (remove bracket part)
df["Release date"] = df["Release date"].str.replace(r"\(.*?\)", "", regex=True).str.strip()

# Convert Release date to datetime (optional but useful)
df["Release date"] = pd.to_datetime(df["Release date"], errors="coerce").dt.to_period("M")

df.head()


,Release date,Actual,Forecast,Previous
0,2026-06,0.0393,0.0400,0.0348
1,2026-05,0.0348,0.0380,0.0340
2,2026-04,0.0340,0.0348,0.0321
3,2026-03,0.0321,0.0310,0.0273
4,2026-02,0.0275,0.0240,0.0133


In [9]:
# Save output
# output_file_path = f"{cwd}..\\data_clean\\cpi.csv"
# df.to_csv(output_file_path, index=False)

print(f"File saved as cpi.csv")


File saved as cpi.csv


Note that in the above raw file the dates in format of 14-Jan-13 needs to be handled manually for which excel has been used.

### Cleaning WPI file

In [10]:
# Read file (works for CSV or TSV)
wpi_file = os.path.join(input_folder, "WPI.xlsx")
df = pd.read_excel(wpi_file)

# Clean column names (just in case)
df.columns = [col.strip() for col in df.columns]

# Clean numeric columns
for col in ["Actual", "Forecast", "Previous"]:
    if col in df.columns:
        df[col] = df[col].apply(clean_percent)

df.head()


,Release date,Actual,Forecast,Previous
0,"Jun 15, 2026 (May)",0.0968,0.0910,0.0830
1,"May 14, 2026 (Apr)",0.0830,0.0440,0.0388
2,"Apr 15, 2026 (Mar)",0.0388,0.0300,0.0213
3,"Mar 16, 2026 (Feb)",0.0213,0.0200,0.0181
4,"Feb 16, 2026 (Jan)",0.0181,0.0125,0.0083


In [11]:
# Clean Release date (remove bracket part)
df["Release date"] = df["Release date"].str.replace(r"\(.*?\)", "", regex=True).str.strip()

# Convert Release date to datetime (optional but useful
df["Release date"] = pd.to_datetime(df["Release date"], errors="coerce").dt.to_period("M")
df.head()


,Release date,Actual,Forecast,Previous
0,2026-06,0.0968,0.0910,0.0830
1,2026-05,0.0830,0.0440,0.0388
2,2026-04,0.0388,0.0300,0.0213
3,2026-03,0.0213,0.0200,0.0181
4,2026-02,0.0181,0.0125,0.0083


In [12]:
# Save output
# output_file_path = f"{cwd}..\\data_clean\\wpi.csv"
# df.to_csv(output_file_path, index=False)

print(f"File saved as wpi.csv")


File saved as wpi.csv


For news & events we need to separate them manually as there is no fixed pattern

In [13]:
clean_folder = os.path.join(os.getcwd(), "..\\data_clean")
files_list = glob.glob(os.path.join(clean_folder, "*.csv"))
for file in files_list:
    df = pd.read_csv(file)
    file_name = os.path.basename(file)
    print(file_name)
    print(df.dtypes)
    print(df.isna().sum())


cpi.csv
release_date     object
actual          float64
forecast        float64
previous        float64
dtype: object
release_date    0
actual          0
forecast        1
previous        0
dtype: int64
market_index.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: int64
nifty_auto.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: int64
nifty_bank.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: int64
nifty_energy.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: int64
nifty_financial_services.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: int64
nifty_fmcg.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: int64
nifty_it.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: int64
nifty_metal.csv
date      object
close    float64
dtype: object
date     0
close    0
dtype: in